In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import tiktoken
from openai import OpenAI
from tqdm.notebook import tqdm
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
# Initialize OpenAI client (Ensure OPENAI_API_KEY is in your environment variables)

import polars as pl

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data'
# DATA_DIR = "../data" # Or your preferred local path
# os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
print("Loading target posts...")
# Load the 7,618 posts first
posts = pl.read_csv(f"{DESTINATION_DIR}/clean_submissions.csv")

# Extract the exact IDs we care about as a Polars Series
valid_post_ids = posts['id'] 
print(f"Targeting {len(valid_post_ids)} unique posts.")

Loading target posts...
Targeting 34867 unique posts.


In [4]:
REQUIRED_COLUMNS = ["id", "body", "link_id", "created_utc"]

file_paths = [
    f"{DESTINATION_DIR}/RAW_comments_todayilearned_part1.parquet",
    f"{DESTINATION_DIR}/RAW_comments_todayilearned_part2.parquet",
    f"{DESTINATION_DIR}/RAW_comments_todayilearned_part3.parquet",
    f"{DESTINATION_DIR}/RAW_comments_todayilearned_part4.parquet",
    f"{DESTINATION_DIR}/RAW_comments_todayilearned_part5.parquet",
    f"{DESTINATION_DIR}/RAW_comments_technology.parquet",
    f"{DESTINATION_DIR}/RAW_comments_philosophy.parquet"
]

all_chunks = []

for path in tqdm(file_paths, desc="Loading & Filtering Parquet files"):
    # Read ONLY required columns from disk
    df_chunk = pl.read_parquet(path, columns=REQUIRED_COLUMNS)
    
    # Clean types and format link_id to match post IDs
    df_chunk = df_chunk.with_columns([
        pl.col("body").cast(pl.String).fill_null(""),
        pl.col("link_id").str.replace("t3_", "").cast(pl.String),
        pl.col("id").cast(pl.String),
        pl.col("created_utc").cast(pl.Int64)
    ])
    
    # INSTANT FILTER: Drop comments that don't belong to our 7,618 posts
    df_chunk = df_chunk.filter(pl.col("link_id").is_in(valid_post_ids))
    
    all_chunks.append(df_chunk)

# Concat the heavily filtered chunks
comments = pl.concat(all_chunks)
all_chunks = None # Free memory immediately
print(f"Total relevant comments loaded: {len(comments)}")

Loading & Filtering Parquet files:   0%|          | 0/7 [00:00<?, ?it/s]

/tmp/ipykernel_10416/3440044012.py:28: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_chunk = df_chunk.filter(pl.col("link_id").is_in(valid_post_ids))


Total relevant comments loaded: 1046369


In [5]:
print("Sorting and extracting top 10 comments per post...")
# Sort by post and time, then grab only the first 10
comments = (
    comments
    .sort(["link_id", "created_utc"])
    .group_by("link_id")
    .head(10)
)

def get_vader_compound(text):
    if not text: return 0.0
    return sia.polarity_scores(str(text))['compound']

print("Calculating early comment sentiment...")
comments = comments.with_columns(
    pl.col("body")
    .map_elements(get_vader_compound, return_dtype=pl.Float64)
    .alias("vader_score")
)

print("Aggregating comment features...")
comment_features = (
    comments.group_by("link_id")
    .agg([
        (pl.len() / 10.0).alias("comment_existence"),
        pl.col("vader_score").mean().alias("avg_early_sentiment"),
        pl.col("vader_score").max().alias("max_early_sentiment"),
        pl.col("vader_score").min().alias("min_early_sentiment")
    ])
)

Sorting and extracting top 10 comments per post...
Calculating early comment sentiment...
Aggregating comment features...


In [6]:
# # Join the engineered features back to the main posts dataframe
reddit = (
    posts
    .join(comment_features, left_on="id", right_on="link_id", how="left")
    .with_columns([
        pl.col('comment_existence').fill_null(0.0),
        pl.col('avg_early_sentiment').fill_null(0.0),
        pl.col('max_early_sentiment').fill_null(0.0),
        pl.col('min_early_sentiment').fill_null(0.0)
    ])
)

# def truncate_text(text):
#     MAX_TOKENS = 8191
#     text = str(text).replace("\n", " ")
#     tokens = encoding.encode(text, disallowed_special=())
#     if len(tokens) > MAX_TOKENS:
#         return encoding.decode(tokens[:MAX_TOKENS])
#     return text

# print("Combining and truncating text for OpenAI embeddings...")
# reddit = reddit.with_columns(
#     pl.col('post')
#     .fill_null("")
#     .map_elements(truncate_text, return_dtype=pl.String)
#     .alias('safe_content')
# )

In [7]:
reddit.head()

allow_live_comments,archived,author,author_fullname,banned_by,category,content_categories,contest_mode,created_utc,discussion_type,distinguished,domain,edited,gilded,hidden,hide_score,id,is_created_from_ads_ui,is_crosspostable,is_meta,is_original_content,is_reddit_media_domain,is_robot_indexable,is_self,is_video,locked,media,media_embed,media_only,name,no_follow,num_comments,num_crossposts,over_18,parent_whitelist_status,permalink,pinned,post_hint,pwls,quarantine,removed_by,removed_by_category,retrieved_on,score,secure_media,secure_media_embed,selftext,send_replies,spoiler,stickied,subreddit_id,subreddit_name_prefixed,subreddit_subscribers,subreddit_type,suggested_sort,title,top_awarded_type,total_awards_received,treatment_tags,upvote_ratio,url,url_overridden_by_dest,view_count,whitelist_status,wls,subreddit,created_utc_dt,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment
str,bool,str,str,str,str,str,bool,i64,str,str,str,str,f64,bool,bool,str,str,bool,str,str,bool,str,bool,bool,bool,str,str,str,str,bool,i64,f64,bool,str,str,str,str,str,str,str,str,f64,i64,str,str,str,bool,bool,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64
null,true,"""Shimaninja""",null,null,null,null,false,1245385801,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8tsj0""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,true,14,0.0,false,"""all_ads""","""/r/philosophy/comments/8tsj0/p…",null,null,null,null,null,null,1.5228e9,0,null,"""{}""","""1) Peoples choices are determi…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""Proof in support of determinis…",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-19 04:30:01""",1.0,0.28394,0.9604,-0.5515
null,true,"""eckermike""",null,null,null,null,false,1245518355,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8u6n7""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,false,149,0.0,false,"""all_ads""","""/r/philosophy/comments/8u6n7/d…",null,null,null,null,null,null,1.5228e9,34,null,"""{}""","""I am in a philosophy class and…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""Did Jesus commit suicide?""",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-20 17:19:15""",1.0,-0.24804,0.5023,-0.9345
null,true,"""Yoshiler""",null,null,null,null,false,1245544563,null,null,"""self.philosophy""","""1245548163.0""",0.0,false,false,"""8u8u8""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,false,173,0.0,false,"""all_ads""","""/r/philosophy/comments/8u8u8/w…",null,null,null,null,null,null,1.5228e9,19,null,"""{}""","""People give up their lives for…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""We are a bunch of stupid apes.""",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-21 00:36:03""",1.0,0.04566,0.89,-0.9476
null,true,"""ibking""",null,null,null,null,false,1245675291,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8ukbf""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,true,75,0.0,false,"""all_ads""","""/r/philosophy/comments/8ukbf/w…",null,null,null,null,null,null,1.5228e9,13,null,"""{}""","""Who do you consider to be inte…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""What does it mean to be 'Inter…",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-22 12:54:51""",1.0,0.32469,0.9711,-0.3182
null,true,"""blackstar9000""",null,null,null,null,false,1245701123,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8uojw""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,true,6,0.0,false,"""all_ads""","""/r/philosophy

In [8]:
reddit.write_csv(f"{DESTINATION_DIR}/reddit_metadata_21_3.csv")

In [ ]:
reddit.head(5)

allow_live_comments,archived,author,author_fullname,banned_by,category,content_categories,contest_mode,created_utc,discussion_type,distinguished,domain,edited,gilded,hidden,hide_score,id,is_created_from_ads_ui,is_crosspostable,is_meta,is_original_content,is_reddit_media_domain,is_robot_indexable,is_self,is_video,locked,media,media_embed,media_only,name,no_follow,num_comments,num_crossposts,over_18,parent_whitelist_status,permalink,pinned,post_hint,pwls,quarantine,removed_by,removed_by_category,retrieved_on,score,secure_media,secure_media_embed,selftext,send_replies,spoiler,stickied,subreddit_id,subreddit_name_prefixed,subreddit_subscribers,subreddit_type,suggested_sort,title,top_awarded_type,total_awards_received,treatment_tags,upvote_ratio,url,url_overridden_by_dest,view_count,whitelist_status,wls,subreddit,created_utc_dt,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment
str,bool,str,str,str,str,str,bool,i64,str,str,str,str,f64,bool,bool,str,str,bool,str,str,bool,str,bool,bool,bool,str,str,str,str,bool,i64,f64,bool,str,str,str,str,str,str,str,str,f64,i64,str,str,str,bool,bool,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64
null,true,"""Shimaninja""",null,null,null,null,false,1245385801,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8tsj0""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,true,14,0.0,false,"""all_ads""","""/r/philosophy/comments/8tsj0/p…",null,null,null,null,null,null,1.5228e9,0,null,"""{}""","""1) Peoples choices are determi…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""Proof in support of determinis…",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-19 04:30:01""",1.0,0.28394,0.9604,-0.5515
null,true,"""eckermike""",null,null,null,null,false,1245518355,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8u6n7""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,false,149,0.0,false,"""all_ads""","""/r/philosophy/comments/8u6n7/d…",null,null,null,null,null,null,1.5228e9,34,null,"""{}""","""I am in a philosophy class and…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""Did Jesus commit suicide?""",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-20 17:19:15""",1.0,-0.24804,0.5023,-0.9345
null,true,"""Yoshiler""",null,null,null,null,false,1245544563,null,null,"""self.philosophy""","""1245548163.0""",0.0,false,false,"""8u8u8""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,false,173,0.0,false,"""all_ads""","""/r/philosophy/comments/8u8u8/w…",null,null,null,null,null,null,1.5228e9,19,null,"""{}""","""People give up their lives for…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""We are a bunch of stupid apes.""",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-21 00:36:03""",1.0,0.04566,0.89,-0.9476
null,true,"""ibking""",null,null,null,null,false,1245675291,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8ukbf""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,true,75,0.0,false,"""all_ads""","""/r/philosophy/comments/8ukbf/w…",null,null,null,null,null,null,1.5228e9,13,null,"""{}""","""Who do you consider to be inte…",true,false,false,"""t5_2qh5b""","""r/philosophy""",null,"""public""",null,"""What does it mean to be 'Inter…",null,null,null,null,"""https://www.reddit.com/r/philo…",null,null,"""all_ads""",null,"""philosophy""","""2009-06-22 12:54:51""",1.0,0.32469,0.9711,-0.3182
null,true,"""blackstar9000""",null,null,null,null,false,1245701123,null,null,"""self.philosophy""","""False""",0.0,false,false,"""8uojw""",null,true,null,null,false,null,true,false,false,null,"""{}""",null,null,true,6,0.0,false,"""all_ads""","""/r/philosophy

In [11]:
reddit.columns

['allow_live_comments',
 'archived',
 'author',
 'author_fullname',
 'banned_by',
 'category',
 'content_categories',
 'contest_mode',
 'created_utc',
 'discussion_type',
 'distinguished',
 'domain',
 'edited',
 'gilded',
 'hidden',
 'hide_score',
 'id',
 'is_created_from_ads_ui',
 'is_crosspostable',
 'is_meta',
 'is_original_content',
 'is_reddit_media_domain',
 'is_robot_indexable',
 'is_self',
 'is_video',
 'locked',
 'media',
 'media_embed',
 'media_only',
 'name',
 'no_follow',
 'num_comments',
 'num_crossposts',
 'over_18',
 'parent_whitelist_status',
 'permalink',
 'pinned',
 'post_hint',
 'pwls',
 'quarantine',
 'removed_by',
 'removed_by_category',
 'retrieved_on',
 'score',
 'secure_media',
 'secure_media_embed',
 'selftext',
 'send_replies',
 'spoiler',
 'stickied',
 'subreddit_id',
 'subreddit_name_prefixed',
 'subreddit_subscribers',
 'subreddit_type',
 'suggested_sort',
 'title',
 'top_awarded_type',
 'total_awards_received',
 'treatment_tags',
 'upvote_ratio',
 'url',
 

In [9]:
print(xxxxxxxxx)

NameError: name 'xxxxxxxxx' is not defined

In [ ]:
# def generate_and_save_embeddings(texts_to_embed, checkpoint_dir="./checkpoints", model="text-embedding-3-small"):
#     MAX_TOKENS_PER_REQUEST = 250000
#     MAX_ROWS_PER_REQUEST = 1000
#     TPM_LIMIT = 900000
#     SAVE_INTERVAL = 5000 
    
#     os.makedirs(checkpoint_dir, exist_ok=True)
    
#     # 1. Dynamic Batching
#     batches = []
#     current_batch = []
#     current_batch_tokens = 0
    
#     for text in texts_to_embed:
#         token_count = len(encoding.encode(text, disallowed_special=()))
#         if current_batch_tokens + token_count > MAX_TOKENS_PER_REQUEST or len(current_batch) >= MAX_ROWS_PER_REQUEST:
#             batches.append((current_batch, current_batch_tokens))
#             current_batch = []
#             current_batch_tokens = 0
#         current_batch.append(text)
#         current_batch_tokens += token_count
        
#     if current_batch:
#         batches.append((current_batch, current_batch_tokens))
        
#     print(f"Divided data into {len(batches)} dynamic batches.")

#     # 2. API Processing
#     current_chunk_embeddings = []
#     chunk_index = 0
#     tokens_in_window = 0
#     window_start_time = time.time()
#     processed_count = 0
    
#     for batch, batch_tokens in tqdm(batches, desc="Fetching Embeddings"):
#         if tokens_in_window + batch_tokens > TPM_LIMIT:
#             elapsed_time = time.time() - window_start_time
#             if elapsed_time < 60:
#                 time.sleep(60 - elapsed_time)
#             tokens_in_window = 0
#             window_start_time = time.time()

#         response = client.embeddings.create(input=batch, model=model)
#         batch_embeddings = [item.embedding for item in response.data]
#         current_chunk_embeddings.extend(batch_embeddings)
#         processed_count += len(batch)
        
#         # Checkpoint Trigger
#         if len(current_chunk_embeddings) >= SAVE_INTERVAL or processed_count == len(texts_to_embed):
#             np.save(f"{checkpoint_dir}/embeddings_part_{chunk_index}.npy", np.array(current_chunk_embeddings))
#             current_chunk_embeddings = []
#             chunk_index += 1

#     # 3. Merge Checkpoints
#     all_files = sorted(glob.glob(f"{checkpoint_dir}/embeddings_part_*.npy"), 
#                        key=lambda x: int(x.split('_part_')[1].split('.npy')[0]))
    
#     embeddings = np.vstack([np.load(f) for f in all_files])
#     print(f"Final Embeddings shape: {embeddings.shape}")
    
#     return embeddings

# # Run the embedding pipeline
# reddit_texts = reddit['safe_content'].to_list()
# reddit_embeddings = generate_and_save_embeddings(
#     reddit_texts, 
#     checkpoint_dir=f"{DESTINATION_DIR}/checkpoints_reddit"
# )

In [ ]:
# reddit_embeddings

In [ ]:
def save_final_features(df, embeddings, output_dir="../data", dataset_prefix="data"):
    # Convert exactly once to Pandas for the easy Numpy boolean indexing
    df_pd = df.to_pandas()
    
    if 'split' not in df_pd.columns:
        np.random.seed(42)
        df_pd['split'] = np.random.choice(['train', 'val', 'test'], size=len(df_pd), p=[0.7, 0.15, 0.15])
    
    for split_name in ['train', 'val', 'test']:
        idx = (df_pd['split'] == split_name).values
        
        X_split = np.hstack([
            df_pd.loc[idx, ['comment_existence']].values,
            embeddings[idx]
        ])
        
        split_filename = f"{output_dir}/{dataset_prefix}_features_{split_name}.npy"
        np.save(split_filename, X_split)
        print(f"[{split_name.upper()}] Saved features shape: {X_split.shape} to {split_filename}")
        
    return df_pd

df_final_results = save_final_features(
    reddit, 
    reddit_embeddings, 
    output_dir=DESTINATION_DIR, 
    dataset_prefix="reddit_final"
)

# Save the full metadata reference frame
df_final_results.to_pickle(f"{DESTINATION_DIR}/reddit_metadata.pkl")
print("Full pipeline executed successfully!")

In [ ]:
df_final_results.columns

In [ ]:
def save_everything_to_pickle(df, embeddings, output_dir="../data", dataset_prefix="data"):
    # Convert exactly once to Pandas
    df_pd = df.to_pandas()
    
    # Convert the numpy array to a list of arrays and assign to a new column
    df_pd['embeddings'] = list(embeddings)
    
    # Save the entire dataframe (including embeddings) to a single pickle
    pickle_filename = f"{output_dir}/{dataset_prefix}_metadata_with_embeddings.pkl"
    df_pd.to_pickle(pickle_filename)
    print(f"Saved full dataframe with embeddings shape: {df_pd.shape} to {pickle_filename}")
    
    return df_pd

# Execute and save directly to pickle
df_final_results = save_everything_to_pickle(
    reddit, 
    reddit_embeddings, 
    output_dir=DESTINATION_DIR, 
    dataset_prefix="reddit"
)
print("Full un-split pipeline executed successfully!")

In [ ]:
import numpy as np

def save_unsplit_features(df, embeddings, output_dir="../data", dataset_prefix="data"):
    # Convert exactly once to Pandas
    df_pd = df.to_pandas()
    
    # Horizontally stack the comment_existence feature with the embeddings for the ENTIRE dataset
    # This keeps your full feature matrix ready for whenever you decide to split it later
    X_full = np.hstack([
        df_pd[['comment_existence']].values,
        embeddings
    ])
    
    # Save the combined full features to a single numpy file
    features_filename = f"{output_dir}/{dataset_prefix}_features_full2.npy"
    np.save(features_filename, X_full)
    print(f"Saved full un-split features shape: {X_full.shape} to {features_filename}")
    
    return df_pd

# Execute the save function without splitting
df_final_results = save_unsplit_features(
    reddit, 
    reddit_embeddings, 
    output_dir=DESTINATION_DIR, 
    dataset_prefix="reddit"
)

# Save the full metadata reference frame to a pickle
df_final_results.to_pickle(f"{DESTINATION_DIR}/reddit_metadata2.pkl")
print("Full un-split pipeline executed successfully!")

In [ ]:
df_final_results.to_csv(f"{DESTINATION_DIR}/reddit_metadata2.csv", index=False)


In [ ]:
print(xxxxxxxxxx)

In [ ]:
def get_vader_compound(text):
    if not text: return 0.0
    return sia.polarity_scores(str(text))['compound']

print("Calculating VADER Sentiment scores...")
# Map the sentiment function to the body column
comments = comments.with_columns(
    pl.col("body")
    .map_elements(get_vader_compound, return_dtype=pl.Float64)
    .alias("vader_score")
)

print("Aggregating features...")
comment_features = (
    comments.group_by("link_id")
    .agg([
        (pl.len() / 10.0).alias("comment_existence"),
        pl.col("vader_score").mean().alias("avg_early_sentiment"),
        pl.col("vader_score").max().alias("max_early_sentiment"),
        pl.col("vader_score").min().alias("min_early_sentiment")
    ])
)

In [ ]:
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def get_vader_compound(text):
    if text is None or text == "":
        return 0.0
    return sia.polarity_scores(str(text))['compound']

def get_early_comments(df_comments, post_id_col='link_id', time_col='created_at'):
    print("Extracting early comments...")
    return (
        df_comments
        .sort([post_id_col, time_col])
        .group_by(post_id_col)
        .head(10)
    )

def engineer_comment_existence(df_early_comments, post_id_col='link_id'):
    print("Calculating Comment Existence...")
    return (
        df_early_comments
        .group_by(post_id_col)
        .agg(pl.len().alias('actual_comment_count'))
        .with_columns((pl.col('actual_comment_count') / 10.0).alias('comment_existence'))
        .select([post_id_col, 'comment_existence'])
    )

def engineer_early_sentiment(df_early_comments, post_id_col='link_id', text_col='body'):
    print("Calculating VADER Sentiment...")
    return (
        df_early_comments
        .with_columns(
            pl.col(text_col)
            .map_elements(get_vader_compound, return_dtype=pl.Float64)
            .alias('vader_score')
        )
        .group_by(post_id_col)
        .agg([
            pl.col('vader_score').mean().alias('avg_early_sentiment'),
            pl.col('vader_score').max().alias('max_early_sentiment'),
            pl.col('vader_score').min().alias('min_early_sentiment')
        ])
    )

In [ ]:
def truncate_text(text):
    MAX_TOKENS = 8191
    text = str(text).replace("\n", " ")
    tokens = encoding.encode(text, disallowed_special=())
    if len(tokens) > MAX_TOKENS:
        return encoding.decode(tokens[:MAX_TOKENS])
    return text

def prepare_embedding_text(df, title_col='title', content_col='content'):
    print("Combining text and enforcing token limits...")
    df_prep = df.copy()
    
    # Combine title and content
    df_prep['text_clean'] = df_prep[title_col].fillna('') + "\n\n" + df_prep[content_col].fillna('')
    
    # Apply token truncation
    tqdm.pandas(desc="Truncating text")
    df_prep['safe_content'] = df_prep['text_clean'].progress_apply(truncate_text)
    
    return df_prep

In [ ]:
def generate_and_save_embeddings(texts_to_embed, checkpoint_dir="./checkpoints", model="text-embedding-3-small"):
    MAX_TOKENS_PER_REQUEST = 250000
    MAX_ROWS_PER_REQUEST = 1000
    TPM_LIMIT = 900000
    SAVE_INTERVAL = 5000 
    
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # 1. Dynamic Batching
    batches = []
    current_batch = []
    current_batch_tokens = 0
    
    for text in texts_to_embed:
        token_count = len(encoding.encode(text, disallowed_special=()))
        
        if current_batch_tokens + token_count > MAX_TOKENS_PER_REQUEST or len(current_batch) >= MAX_ROWS_PER_REQUEST:
            batches.append((current_batch, current_batch_tokens))
            current_batch = []
            current_batch_tokens = 0
            
        current_batch.append(text)
        current_batch_tokens += token_count
        
    if current_batch:
        batches.append((current_batch, current_batch_tokens))
        
    print(f"Divided data into {len(batches)} dynamic batches.")

    # 2. API Processing
    current_chunk_embeddings = []
    chunk_index = 0
    tokens_in_window = 0
    window_start_time = time.time()
    processed_count = 0
    
    for batch, batch_tokens in tqdm(batches, desc="Fetching Embeddings"):
        if tokens_in_window + batch_tokens > TPM_LIMIT:
            elapsed_time = time.time() - window_start_time
            if elapsed_time < 60:
                sleep_time = 60 - elapsed_time
                print(f"  [Rate Limit Paused] Sleeping for {sleep_time:.2f} seconds...")
                time.sleep(sleep_time)
            
            tokens_in_window = 0
            window_start_time = time.time()

        response = client.embeddings.create(input=batch, model=model)
        
        tokens_in_window += batch_tokens
        batch_embeddings = [item.embedding for item in response.data]
        current_chunk_embeddings.extend(batch_embeddings)
        processed_count += len(batch)
        
        # Checkpoint Trigger
        if len(current_chunk_embeddings) >= SAVE_INTERVAL or processed_count == len(texts_to_embed):
            chunk_array = np.array(current_chunk_embeddings)
            filename = f"{checkpoint_dir}/embeddings_part_{chunk_index}.npy"
            np.save(filename, chunk_array)
            current_chunk_embeddings = []
            chunk_index += 1

    # 3. Merge Checkpoints
    all_files = sorted(glob.glob(f"{checkpoint_dir}/embeddings_part_*.npy"), 
                       key=lambda x: int(x.split('_part_')[1].split('.npy')[0]))
    
    embeddings = np.vstack([np.load(f) for f in all_files])
    print(f"Final Embeddings shape: {embeddings.shape}")
    
    return embeddings

    
def save_final_features(df, embeddings, output_dir="../data", dataset_prefix="data"):
    # Convert Polars DF to Pandas just for the final split logic to keep your numpy code identical
    df_final = df.to_pandas() 
    
    if 'split' not in df_final.columns:
        np.random.seed(42)
        df_final['split'] = np.random.choice(['train', 'val', 'test'], size=len(df_final), p=[0.7, 0.15, 0.15])
    
    # Save the splits to numpy files
    for split_name in ['train', 'val', 'test']:
        idx = (df_final['split'] == split_name).values
        
        # Combine the existence fraction + the embedding dimensions
        X_split = np.hstack([
            df_final.loc[idx, ['comment_existence']].values,
            embeddings[idx]
        ])
        
        split_filename = f"{output_dir}/{dataset_prefix}_features_{split_name}.npy"
        np.save(split_filename, X_split)
        print(f"[{split_name.upper()}] Saved features (shape: {X_split.shape}) to {split_filename}")
        
    return df_final

In [ ]:
print(xxxxxxxxxxxx)

In [ ]:
# 1. Pre-process link_id to match post id (e.g., 't3_abc123' -> 'abc123')
comments = comments.with_columns(
    pl.col("link_id").str.replace("t3_", "")
)

# 2. Run Pipeline
df_first_10 = get_early_comments(comments)
existence_feat = engineer_comment_existence(df_first_10)
sentiment_feat = engineer_early_sentiment(df_first_10)

# 3. Merge with Posts (Joining on link_id == id)
df_final = (
    posts
    .join(existence_feat, left_on="id", right_on="link_id", how="left")
    .join(sentiment_feat, left_on="id", right_on="link_id", how="left")
    .with_columns([
        pl.col('comment_existence').fill_null(0.0),
        pl.col('avg_early_sentiment').fill_null(0.0),
        pl.col('max_early_sentiment').fill_null(0.0),
        pl.col('min_early_sentiment').fill_null(0.0)
    ])
)

# 4. Truncate text for OpenAI
print("Truncating text...")
df_final = df_final.with_columns(
    pl.col('post')
    .fill_null("")
    .map_elements(truncate_text, return_dtype=pl.String)
    .alias('safe_content')
)

print("Process complete. No Pandas overhead used!")

In [ ]:
pd_dfs = [
    comments1.to_pandas(), 
    comments2.to_pandas(), 
    comments3.to_pandas(), 
    comments4.to_pandas(), 
    comments5.to_pandas(), 
    comments6.to_pandas(), 
    comments7.to_pandas()
]

# Pandas will handle the Null vs String issue automatically
combined_pd = pd.concat(pd_dfs, ignore_index=True)

In [ ]:


nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def get_vader_compound(text):
    if pd.isna(text) or str(text).strip() == "":
        return 0.0
    return sia.polarity_scores(str(text))['compound']

def get_early_comments(df_comments, post_id_col='post_id', time_col='created_at'):
    """Sorts and extracts only the first 10 comments for each post."""
    print("Extracting early comments...")
    df_comm = df_comments.copy()
    df_comm[time_col] = pd.to_datetime(df_comm[time_col])
    
    df_early = df_comm.sort_values([post_id_col, time_col])
    return df_early.groupby(post_id_col).head(10).copy()

def engineer_comment_existence(df_early_comments, post_id_col='post_id'):
    """Calculates the Comment Existence fraction (0.0 to 1.0)."""
    print("Calculating Comment Existence feature...")
    
    existence_features = df_early_comments.groupby(post_id_col).agg(
        actual_comment_count=('id', 'count')
    )
    existence_features['comment_existence'] = existence_features['actual_comment_count'] / 10.0
    
    return existence_features[['comment_existence']]

def engineer_early_sentiment(df_early_comments, post_id_col='post_id', text_col='content'):
    """Calculates VADER sentiment aggregates for the early comments."""
    print("Calculating VADER Sentiment features...")
    
    # Apply VADER scoring
    df_early_comments['vader_score'] = df_early_comments[text_col].apply(get_vader_compound)
    
    # Aggregate mean, max, and min
    sentiment_features = df_early_comments.groupby(post_id_col).agg(
        avg_early_sentiment=('vader_score', 'mean'),
        max_early_sentiment=('vader_score', 'max'),
        min_early_sentiment=('vader_score', 'min')
    )
    
    return sentiment_features

def merge_engineered_features(df_posts, existence_df, sentiment_df, post_id_col_posts='id'):
    """Merges all engineered features back into the main posts dataframe and handles NaNs."""
    print("Merging features into main posts dataframe...")
    df_merged = df_posts.copy()
    
    # Merge Existence
    df_merged = df_merged.merge(
        existence_df, left_on=post_id_col_posts, right_index=True, how='left'
    )
    df_merged['comment_existence'] = df_merged['comment_existence'].fillna(0.0)
    
    # Merge Sentiment
    df_merged = df_merged.merge(
        sentiment_df, left_on=post_id_col_posts, right_index=True, how='left'
    )
    fill_cols = ['avg_early_sentiment', 'max_early_sentiment', 'min_early_sentiment']
    df_merged[fill_cols] = df_merged[fill_cols].fillna(0.0)
    
    return df_merged

In [ ]:
print(xxxxxxxxxx)

In [ ]:
comments["created_at"] = pd.to_datetime(comments["created_utc"], unit='s')
posts["created_at"] = pd.to_datetime(posts["created_utc"], unit='s')

In [ ]:
df_first_10 = get_early_comments(
    comments, 
    post_id_col='post_id', 
    time_col='created_at'
)

# 2. Generate the distinct feature sets using the isolated dataframe
existence_features = engineer_comment_existence(
    df_first_10, 
    post_id_col='post_id'
)

sentiment_features = engineer_early_sentiment(
    df_first_10, 
    post_id_col='post_id', 
    text_col='content'
)

# 3. Merge everything back to your main posts dataframe
df_moltbook_features = merge_engineered_features(
    df_ai_posts_final, 
    existence_features, 
    sentiment_features, 
    post_id_col_posts='id'
)

In [ ]:
df_moltbook_features

In [ ]:
df_moltbook_features.to_csv(f"{DATA_DIR}/moltbook_features_1.csv", index=False)

In [ ]:
df_moltbook_features.columns

In [ ]:
tqdm.pandas(desc="Truncating text")
df_moltbook_features['safe_content'] = df_moltbook_features['post'].progress_apply(truncate_text)

In [ ]:
print(xxxxxx)

In [ ]:
# df_moltbook = prepare_embedding_text(df_moltbook_features, title_col='title', content_col='content')
moltbook_texts = df_moltbook_features['safe_content'].tolist()

In [ ]:
moltbook_embeddings = generate_and_save_embeddings(
    moltbook_texts, 
    checkpoint_dir=f"{DATA_DIR}/checkpoints_moltbook"
)

In [ ]:
df_moltbook_final = save_final_features(
    df_moltbook_features, 
    moltbook_embeddings, 
    output_dir=DATA_DIR, 
    dataset_prefix="moltbook_mehmeh"
)

In [ ]:
df_moltbook_final.to_pickle(f"{DATA_DIR}/moltbook_final_mehmeh.pkl")

In [ ]:
df_moltbook_final

In [ ]:
print(xxxxxxxxxx)

In [ ]:
# 1. Feature Engineering


# 2. Text Preparation
df_moltbook = prepare_embedding_text(df_moltbook, title_col='title', content_col='content')
moltbook_texts = df_moltbook['safe_content'].tolist()

# 3. Fetch Embeddings
moltbook_embeddings = generate_and_save_embeddings(
    moltbook_texts, 
    checkpoint_dir="./checkpoints_moltbook"
)

# 4. Save Final Matrices
df_moltbook_final = save_final_features(
    df_moltbook, 
    moltbook_embeddings, 
    output_dir="../data", 
    dataset_prefix="moltbook"
)